In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
SPLITS = Path("../01_data/interim/splits")
TABLES = Path("../04_outputs/tables")
CHECKPOINTS = Path("../03_models/checkpoints")

TABLES.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01

In [4]:
train_df = pd.read_csv(SPLITS / "banglasarc3_ternary_train.csv")
val_df = pd.read_csv(SPLITS / "banglasarc3_ternary_val.csv")
test_df = pd.read_csv(SPLITS / "banglasarc3_ternary_test.csv")

print(train_df.shape, val_df.shape, test_df.shape)
display(train_df.head())

(9657, 4) (1207, 4) (1208, 4)


,text,label_ternary,label_original,dataset_name
0,এইলা দুঃক্ষের কথা আর না কইস রে ভাই,2,Sarcastic,banglasarc3
1,সফল এবং ধনী ব্যক্তিরা সব সময় নিজেদের কাজে মনোয...,1,Neutral,banglasarc3
2,বাংলাদেশের শেষ প্রান্তে দাঁড়ানোর সৌভাগ্যবান হ...,1,Neutral,banglasarc3
3,আহা ঠকে গেলাম তো জীবন যুদ্ধে পিছিয়ে গেলাম,2,Sarcastic,banglasarc3
4,কল্পনার নতুন শুরু হোক সুখের এবং শান্তিময়,0,Non-Sarcastic,banglasarc3


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [6]:
train_df = train_df[["text", "label_ternary"]].rename(columns={"label_ternary": "label"})
val_df = val_df[["text", "label_ternary"]].rename(columns={"label_ternary": "label"})
test_df = test_df[["text", "label_ternary"]].rename(columns={"label_ternary": "label"})

In [8]:
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)

In [9]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

Map: 100%|██████████| 1208/1208 [00:00<00:00, 21318.56 examples/s]


In [10]:
train_ds = train_ds.remove_columns(["text"])
val_ds = val_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    return {
        "accuracy": acc,
        "macro_f1": f1_macro,
        "weighted_f1": f1_weighted,
    }

In [12]:
class_weights = torch.tensor([1.15, 1.00, 1.00], dtype=torch.float)
class_weights

tensor([1.1500, 1.0000, 1.0000])

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 55559.30it/s]
ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	

In [14]:
class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            weight = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weight)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [15]:
training_args = TrainingArguments(
    output_dir="../03_models/checkpoints/banglabert_weighted_banglasarc3_ternary",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

In [16]:
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

In [17]:
trainer.train()

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.890330,0.823947,0.634631,0.617734,0.617985
2,0.665588,0.818857,0.656172,0.657688,0.657831


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.97it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.62it/s]
There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.Lay

TrainOutput(global_step=2416, training_loss=0.7779593562448262, metrics={'train_runtime': 1382.9889, 'train_samples_per_second': 13.965, 'train_steps_per_second': 1.747, 'total_flos': 1270443137499648.0, 'train_loss': 0.7779593562448262, 'epoch': 2.0})

In [18]:
def predict_metrics(trainer, df_for_labels, ds):
    output = trainer.predict(ds)
    preds = np.argmax(output.predictions, axis=-1)
    labels = np.array(df_for_labels["label"])

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    cm = confusion_matrix(labels, preds)

    return {
        "accuracy": float(acc),
        "macro_f1": float(f1_macro),
        "weighted_f1": float(f1_weighted),
    }, cm, labels, preds

val_metrics, val_cm, _, _ = predict_metrics(trainer, val_df, val_ds)
test_metrics, test_cm, test_labels, test_preds = predict_metrics(trainer, test_df, test_ds)

print("Validation metrics:", val_metrics)
print("Validation confusion matrix:\n", val_cm)

print("\nTest metrics:", test_metrics)
print("Test confusion matrix:\n", test_cm)

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Validation metrics: {'accuracy': 0.656172328086164, 'macro_f1': 0.6576875194522253, 'weighted_f1': 0.6578309921041048}
Validation confusion matrix:
 [[238  87  76]
 [ 99 286  20]
 [ 95  38 268]]

Test metrics: {'accuracy': 0.652317880794702, 'macro_f1': 0.6528759971969852, 'weighted_f1': 0.6529814274027869}
Test confusion matrix:
 [[238  89  74]
 [ 90 271  45]
 [ 89  33 279]]


In [19]:
label_names = {
    0: "Non-Sarcastic",
    1: "Neutral",
    2: "Sarcastic",
}

print(classification_report(
    test_labels,
    test_preds,
    target_names=[label_names[0], label_names[1], label_names[2]],
    zero_division=0
))

               precision    recall  f1-score   support

Non-Sarcastic       0.57      0.59      0.58       401
      Neutral       0.69      0.67      0.68       406
    Sarcastic       0.70      0.70      0.70       401

     accuracy                           0.65      1208
    macro avg       0.65      0.65      0.65      1208
 weighted avg       0.65      0.65      0.65      1208



In [20]:
results = [
    {
        "model": "banglabert_weighted",
        "dataset": "banglasarc3_ternary",
        "split": "validation",
        "accuracy": val_metrics["accuracy"],
        "macro_f1": val_metrics["macro_f1"],
        "weighted_f1": val_metrics["weighted_f1"],
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "max_length": MAX_LENGTH,
        "seed": SEED,
        "class_weights": str(class_weights.tolist()),
    },
    {
        "model": "banglabert_weighted",
        "dataset": "banglasarc3_ternary",
        "split": "test",
        "accuracy": test_metrics["accuracy"],
        "macro_f1": test_metrics["macro_f1"],
        "weighted_f1": test_metrics["weighted_f1"],
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "max_length": MAX_LENGTH,
        "seed": SEED,
        "class_weights": str(class_weights.tolist()),
    },
]

results_df = pd.DataFrame(results)
results_df.to_csv(TABLES / "banglabert_weighted_banglasarc3_ternary_results.csv", index=False)

with open(TABLES / "banglabert_weighted_banglasarc3_ternary_confusion_matrices.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "validation": val_cm.tolist(),
            "test": test_cm.tolist(),
        },
        f,
        ensure_ascii=False,
        indent=2
    )

results_df

,model,dataset,split,accuracy,macro_f1,weighted_f1,epochs,batch_size,learning_rate,max_length,seed,class_weights
0,banglabert_weighted,banglasarc3_ternary,validation,0.656172,0.657688,0.657831,2,8,0.00002,128,42,"[1.149999976158142, 1.0, 1.0]"
1,banglabert_weighted,banglasarc3_ternary,test,0.652318,0.652876,0.652981,2,8,0.00002,128,42,"[1.149999976158142, 1.0, 1.0]"
